In [ ]:
# ============================================================================
# 1. 설정 및 라이브러리
# ============================================================================

import os
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone
from openai import OpenAI
import re

# .env 파일 로드
env_path = Path(__file__).parent / '.env' if '__file__' in globals() else Path.cwd() / '.env'
load_dotenv(env_path)

# 환경 변수에서 API 키 로드
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
INDEX_NAME = "target"  # target 인덱스 사용

# 클라이언트 초기화
pc = Pinecone(api_key=PINECONE_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)
index = pc.Index(INDEX_NAME)

print(f"[완료] Pinecone '{INDEX_NAME}' 인덱스 연결 완료")
print(index.describe_index_stats())

In [2]:
# ============================================================================
# 2. 유틸리티 함수
# ============================================================================

def get_embedding(text: str, model: str = "text-embedding-3-large") -> list:
    """OpenAI 임베딩 생성"""
    text = text.replace("\n", " ").strip()
    if not text:
        return None
    
    response = openai_client.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding


def parse_ipc_codes(ipc_string: str) -> list:
    """
    IPC 코드 문자열을 파싱하여 상위 클래스 리스트 반환
    예: "A61K 31/501|A61P 9/04" -> ["A61K", "A61P"]
    """
    if not ipc_string:
        return []
    
    # IPC 코드 패턴: 대문자+숫자+대문자 (예: A61K, B32B, E06B)
    pattern = r'([A-Z]\d{2}[A-Z])'
    matches = re.findall(pattern, ipc_string)
    
    # 중복 제거하고 리스트로 반환
    return list(set(matches))


def create_search_text(abstract: str, invention_name: str) -> str:
    """검색용 텍스트 생성 (초록 + 특허명 결합)"""
    parts = []
    if invention_name:
        parts.append(f"특허명: {invention_name}")
    if abstract:
        parts.append(f"초록: {abstract}")
    return " ".join(parts)


print("[완료] 유틸리티 함수 정의 완료")

✅ 유틸리티 함수 정의 완료


In [6]:
# ============================================================================
# 3. 유사 특허 검색 함수
# ============================================================================

def find_similar_patents(
    ipc_code: str,
    abstract: str,
    invention_name: str,
    top_k: int = 5,
    use_ipc_filter: bool = True
) -> list:
    """
    IPC 코드로 1차 필터링 후, 초록+특허명 유사도로 검색
    
    Args:
        ipc_code: IPC 코드 문자열 (예: "A61K 31/501|A61P 9/04")
        abstract: 초록
        invention_name: 특허명
        top_k: 반환할 결과 수 (기본 5)
        use_ipc_filter: IPC 필터링 사용 여부 (기본 True)
    
    Returns:
        유사한 특허 리스트
    """
    
    # 1. 검색용 텍스트 생성 및 임베딩
    search_text = create_search_text(abstract, invention_name)
    query_embedding = get_embedding(search_text)
    
    if not query_embedding:
        print("[오류] 임베딩 생성 실패")
        return []
    
    # 2. IPC 코드 파싱 (상위 클래스 추출)
    ipc_classes = parse_ipc_codes(ipc_code)
    print(f"[정보] 입력 IPC 클래스: {ipc_classes}")
    
    # 3. Pinecone 검색 (IPC 필터링 적용)
    filter_condition = None
    
    if use_ipc_filter and ipc_classes:
        # IPC 코드가 포함된 항목만 필터링
        # $or 조건으로 여러 IPC 클래스 중 하나라도 포함되면 매칭
        filter_condition = {
            "$or": [
                {"ipc_code": {"$contains": ipc_class}} 
                for ipc_class in ipc_classes
            ]
        }
        print(f"[검색] IPC 필터 적용: {ipc_classes}")
    
    # 4. 유사도 검색 실행
    try:
        results = index.query(
            vector=query_embedding,
            top_k=top_k * 2 if use_ipc_filter else top_k,  # 필터링 시 더 많이 검색
            include_metadata=True,
            filter=filter_condition
        )
    except Exception as e:
        # 필터 실패 시 필터 없이 재시도
        print(f"[경고] IPC 필터링 실패, 필터 없이 검색: {e}")
        results = index.query(
            vector=query_embedding,
            top_k=top_k,
            include_metadata=True
        )
    
    # 5. 결과 정리
    similar_patents = []
    for match in results.matches[:top_k]:
        patent = {
            "score": match.score,
            "id": match.id,
            "target_short_name": match.metadata.get("target_short_name", "N/A"),
            "target_id": match.metadata.get("target_id", "N/A"),
            "invention_name": match.metadata.get("invention_name", "N/A"),
            "abstract": match.metadata.get("abstract", "N/A"),
            "ipc_code": match.metadata.get("ipc_code", "N/A"),
            "application_number": match.metadata.get("application_number", "N/A"),
            "application_date": match.metadata.get("application_date", "N/A"),
        }
        similar_patents.append(patent)
    
    return similar_patents


def print_results(results: list):
    """검색 결과를 보기 좋게 출력"""
    if not results:
        print("[오류] 검색 결과 없음")
        return
    
    print(f"\n{'='*80}")
    print(f"[결과] Target 유사 특허 검색 결과 (상위 {len(results)}개)")
    print(f"{'='*80}\n")
    
    for i, patent in enumerate(results, 1):
        print(f"[{i}] 유사도: {patent['score']:.4f}")
        print(f"    Target: {patent['target_short_name']} (ID: {patent['target_id']})")
        print(f"     특허명: {patent['invention_name'][:80]}...")
        print(f"     IPC: {patent['ipc_code'][:50]}...")
        print(f"     초록: {patent['abstract'][:150]}...")
        print(f"     출원번호: {patent['application_number']}")
        print(f"    출원일: {patent['application_date']}")
        print(f"    {'-'*70}")
    
    print()


print("[완료] 검색 함수 정의 완료")

[완료] 검색 함수 정의 완료


In [7]:
# ============================================================================
# 4. 사용 예시
# ============================================================================

# 검색할 특허 정보 입력
input_patent = {
    "ipc_code": "A61K 39/05",
    "invention_name": "PROPIONIBACTERIUM ACNES PROPHYLACTIC AND THERAPEUTIC IMMUNE TREATMENT",  # 특허명
    "abstract": "The present invention discloses a vaccine comprising one or more of Dermatan sulfate-binding adhesin 1 of P. acnes (DsA1 polypeptide), Dermatan sulfate-binding adhesin 2 of P. acnes (DsA2 polypeptide), and putative iron-transport protein (PITP) polypeptide of P. acnes, and/or a fragment and/or derivative of DsA1 and/or DsA2 and/or PITP, wherein the DsA1 polypeptide and the DsA2 polypeptide comprise from N- to C-terminus an N-terminal swapping region (“NSR”), a first conserved sub-domain (“CSD1”), a first swapping region (“SR1”), a second conserved sub-domain (“CSD2”), a second swapping region (“SR2”), a third conserved sub-domain (“CSD3”), a Pro-Thr repeat containing region (“PT repeat region”), and a C-terminal region (“CTR”), and wherein the PITP polypeptide comprises from N- to C-terminus an extended neocarzinostatin family domain (“ENFD”), a first swapping region (“SR1”), a heme-binding domain (“HbD”), a second swapping region (“SR2”) including the C-terminal LPXTG motif, and a hydrophobic C-terminal region (“HLAR”)."
}

print("📥 입력 특허 정보:")
print(f"   IPC: {input_patent['ipc_code']}")
print(f"   특허명: {input_patent['invention_name']}")
print(f"   초록: {input_patent['abstract'][:100]}...")
print()

# 유사 특허 검색 실행
results = find_similar_patents(
    ipc_code=input_patent["ipc_code"],
    abstract=input_patent["abstract"],
    invention_name=input_patent["invention_name"],
    top_k=5,
    use_ipc_filter=True  # IPC 필터링 사용
)

# 결과 출력
print_results(results)

📥 입력 특허 정보:
   IPC: A61K 39/05
   특허명: PROPIONIBACTERIUM ACNES PROPHYLACTIC AND THERAPEUTIC IMMUNE TREATMENT
   초록: The present invention discloses a vaccine comprising one or more of Dermatan sulfate-binding adhesin...

[정보] 입력 IPC 클래스: ['A61K']
[검색] IPC 필터 적용: ['A61K']
[경고] IPC 필터링 실패, 필터 없이 검색: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 09 Feb 2026 07:34:00 GMT', 'Content-Type': 'application/json', 'Content-Length': '69', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '430', 'x-envoy-upstream-service-time': '35', 'x-pinecone-response-duration-ms': '431', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"$contains is not a valid operator","details":[]}


[결과] Target 유사 특허 검색 결과 (상위 5개)

[1] 유사도: 0.8455
    Target: ORIGIMM Biotechnology GmbH (ID: 41154)
     특허명: PROPIONIBACTERIUM ACNES PROPHYLACTIC AND THERAPEUTIC IMMUNE TREATMENT...
     IPC: A61K39/05|A61P17/10|A61P31/04|C07K14/195...
     초록:  The present invention discl

In [7]:
# ============================================================================
# 6. Acquiror 인덱스 연결
# ============================================================================

ACQUIROR_INDEX_NAME = "acquiror"

# Acquiror 인덱스 연결
acquiror_index = pc.Index(ACQUIROR_INDEX_NAME)

print(f"[완료] Pinecone '{ACQUIROR_INDEX_NAME}' 인덱스 연결 완료")
print(acquiror_index.describe_index_stats())

✅ Pinecone 'acquiror' 인덱스 연결 완료
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '192',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 26 Jan 2026 08:21:42 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '39',
                                    'x-pinecone-request-id': '6738672072312535630',
                                    'x-pinecone-request-latency-ms': '38',
                                    'x-pinecone-response-duration-ms': '40'}},
 'dimension': 3072,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 114600}},
 'storageFullness': 0.0,
 'total_vector_count': 114600,
 'vector_type': 'dense'}


✅ Acquiror 검색 함수 정의 완료


In [9]:
# ============================================================================
# 8. Acquiror 검색 사용 예시
# ============================================================================

# 검색할 특허 정보 입력
input_patent_acquiror = {
    "ipc_code": "G06Q 30/00",
    "invention_name": "유무형 상품 결합을 통한 온라인 판매 방법 및 시스템",  # 특허명
    "abstract": "본 발명은 온라인 쇼핑몰에서 판매하는 실제의 유형상품과 무형상품에 대한 활동을 결합한 상품에 대하여 다양한 수단을 이용하여 추천한 회원에게 기존의 가격보다 낮은 가격으로 판매하는 방법 및 시스템에 관한 것으로, 고객 에 의해 선택된 적어도 하나의 유형상품에 대한 주문서 작성을 입력받는 단계와, 상기 고객의 관련 정보를 근거로 상기 유형상품에 할인혜택 적용여부를 선택할 수 있는 화면을 출력하는 단계와, 상기 할인혜택에 대하여 추천 회원들로부터 작성된 추천 게시물을 출력하는 단계와, 상기 할인혜택과 관련된 무형상품 및 상기 게시물을 작성한 추천회원에 대한 상기 고객의 입력을 수신하는 단계와, 상기 고객의 입력을 근거로 한 할인혜택을 상기 유형 상품 가격에 적용하는 단계 및 상기 추천회원에게 소정의 혜택을 부여하는 단계를 포함하여 이루어질 수 있다."
}

print("📥 입력 특허 정보 (Acquiror 검색):")
print(f"   IPC: {input_patent_acquiror['ipc_code']}")
print(f"   특허명: {input_patent_acquiror['invention_name']}")
print(f"   초록: {input_patent_acquiror['abstract'][:100]}...")
print()

# Acquiror 인덱스에서 유사 특허 검색 실행
acquiror_results = find_similar_acquiror_patents(
    ipc_code=input_patent_acquiror["ipc_code"],
    abstract=input_patent_acquiror["abstract"],
    invention_name=input_patent_acquiror["invention_name"],
    top_k=5,
    use_ipc_filter=True  # IPC 필터링 사용
)

# 결과 출력
print_acquiror_results(acquiror_results)

📥 입력 특허 정보 (Acquiror 검색):
   IPC: G06Q 30/00
   특허명: 유무형 상품 결합을 통한 온라인 판매 방법 및 시스템
   초록: 본 발명은 온라인 쇼핑몰에서 판매하는 실제의 유형상품과 무형상품에 대한 활동을 결합한 상품에 대하여 다양한 수단을 이용하여 추천한 회원에게 기존의 가격보다 낮은 가격으로 판매하는 ...

📌 입력 IPC 클래스: ['G06Q']
🔍 IPC 필터 적용됨: ['G06Q'] (후처리)

🎯 Acquiror 유사 특허 검색 결과 (상위 5개)

[1] 유사도: 0.6645
    🏢 Acquiror: Coupang Corp (ID: 9206)
    🌍 Nation: South Korea
    📋 특허명: 쇼핑 서비스 제공 시스템 및 쇼핑 서비스 제공 방법...
    🏷️ IPC: G06Q 30/06|G06Q 30/02...
    📝 초록: 본 발명은 쇼핑 서비스 제공 시스템 및 쇼핑 서비스 제공 방법에 관한 것이다.  본 발명의 일 실시 예에 따른 쇼핑 서비스 제공 시스템은 쇼핑 서비스를 제공하는 사이트에 접속한 사용자의 상품 구매 이력으로부터 쓰는 대로 닳거나 줄어들어 없어지거나 못쓰게 되는 소모품에 ...
    📄 출원번호: 1020150048322
    📅 출원일: 20150406
    ----------------------------------------------------------------------
[2] 유사도: 0.6497
    🏢 Acquiror: Bus Insight Inc (ID: 6217)
    🌍 Nation: South Korea
    📋 특허명: 온라인 상거래에 대한 오프라인 인센티브 부여(또는 매출 귀속) 커머스 모델...
    🏷️ IPC: G06Q 30/06|G06K 19/06|G06Q 30/02...
    📝 초록: 온·오프라인 연계 상거래 방법 및 그 장치가 개시된다. 본 발명의 일 실시 예에 따른 온·오프라인 연계 상거래 방법은, 소비자가 오프